# MODEL SELECTION AND MODEL SAVING
Seattle Weather Prediction Project

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

## 2. Load Dataset

In [ ]:
df = pd.read_csv("seattle-weather.csv")

print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

## 3. Date Feature Engineering

In [ ]:
df['date'] = pd.to_datetime(df['date'])

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

df.drop('date', axis=1, inplace=True)

print("Updated Columns:", df.columns.tolist())

## 4. Separate Features (X) and Target (y)

In [ ]:
X = df.drop(columns=['weather'])
y = df['weather']

print("Features:", X.columns.tolist())
print("Weather Classes:", y.unique())

## 5. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)

## 6. Define Models

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )
}

results = []

## 7. Train and Evaluate Models

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Macro F1": f1
    })

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print("Accuracy:", round(accuracy * 100, 2), "%")
    print("Macro F1:", round(f1, 4))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

## 8. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

print(results_df)

## 9. Select Initial Best Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]

print("Best Initial Model:", best_model_name)
print("Best Initial Accuracy:", round(results_df.iloc[0]["Accuracy"] * 100, 2), "%")

## 10. Cross-Validation

In [ ]:
print("5-FOLD CROSS VALIDATION")

for name, model in models.items():
    cv_scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="accuracy"
    )

    print(
        name,
        "| Mean CV Accuracy:",
        round(cv_scores.mean() * 100, 2),
        "%"
    )

## 11. Hyperparameter Tuning - Random Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", round(grid.best_score_ * 100, 2), "%")

## 12. Final Best Model Evaluation

In [ ]:
best_model = grid.best_estimator_

best_model.fit(X_train, y_train)

final_y_pred = best_model.predict(X_test)

print("Final Test Accuracy:", round(accuracy_score(y_test, final_y_pred) * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, final_y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, final_y_pred))

## 13. Save Final Model File

In [ ]:
model_data = {
    'model': best_model,
    'features': X.columns.tolist()
}

joblib.dump(model_data, 'weather_prediction_model.pkl')

print("Model saved successfully!")
print("File Name: weather_prediction_model.pkl")

## 14. Load Saved Model

In [ ]:
loaded_data = joblib.load('weather_prediction_model.pkl')

loaded_model = loaded_data['model']
feature_names = loaded_data['features']

print("Model loaded successfully!")
print("Required Features:", feature_names)

## 15. Final Prediction

In [ ]:
new_data = pd.DataFrame({
    'precipitation': [0.0],
    'temp_max': [20.0],
    'temp_min': [12.0],
    'wind': [2.0],
    'year': [2015],
    'month': [6],
    'day': [15]
})

# Ensure exact same feature order as training
new_data = new_data[feature_names]

prediction = loaded_model.predict(new_data)

print("Predicted Weather:", prediction[0])

## Project Flow Complete
Dataset → Feature Engineering → Train/Test Split → Model Comparison → Cross Validation → Hyperparameter Tuning → Final Evaluation → Model Saving (.pkl) → Model Loading → Prediction